# A whole run against AMW: quarters, and our months

Shows what `compare_run.py` wrote for a run (on the cluster: `<run>/amw_comparison/`). Each
row of `chips_<quarter>.parquet` is a chip that someone detected (score > 0.4): our quarter,
AMW's quarter, or one of our three months. AMW has no months, so our months are compared with
AMW's quarter. Area is left out.

Set `RESULTS` below, then *Run All*. Reads only small files; runs on the laptop.

In [ ]:
RESULTS = "tests_output/aca_full_2026-10-23/amw_comparison"   # copied from the cluster (README)
# RESULTS = "/work/goldmine/grambauer/runs/aca_full_2026-10-23/amw_comparison"   # on Goethe

import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "panel_pipeline").is_dir())
sys.path.insert(0, str(REPO / "amw_comparison"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import compare

RESULTS = Path(RESULTS) if Path(RESULTS).is_absolute() else REPO / RESULTS
pd.set_option("display.float_format", "{:,.5f}".format)
chips = {p.stem.removeprefix("chips_"): pd.read_parquet(p) for p in sorted(RESULTS.glob("chips_*.parquet"))}
QUARTERS = list(chips)
for q, d in chips.items():
    d["outcome"] = compare.outcome(d)
print(f"quarters: {QUARTERS}")
chips[QUARTERS[-1]].head()

## 1. Inputs

Every task read must be complete with no failed tiles; a missing tile looks like "AMW detected
it, we didn't". `should have matched` must be 0, or the chip grids differ.

In [ ]:
inputs = pd.read_csv(RESULTS / "inputs.csv")
problems = inputs[(inputs["state"] != "complete") | (inputs["failed"].fillna(0) > 0)]
print(f"{len(inputs)} tasks, {len(problems)} not complete or with failed tiles")
display(problems)
joins = pd.read_csv(RESULTS / "joins.csv")
assert joins["should have matched"].sum() == 0, "AMW chips fell between our chips: the grids differ"
joins.pivot(index="subregion", columns="quarter", values="AMW rows on our chips")

## 2. Our quarters against AMW's

Chips above 0.4. *Overlap* = chips both detected / chips either detected (1 = the same set).
Border tiles count once.

In [ ]:
summary = pd.DataFrame({q: compare.agreement(d) for q, d in chips.items()}).T.drop(columns="neither")
whole = ["both", "only ours", "only AMW", "ours total", "AMW total"]
summary[whole] = summary[whole].astype(int)
summary

In [ ]:
# the same, per subregion (a border chip counts in the subregion whose score was kept)
per_sub = pd.concat({q: d.groupby("subregion").apply(compare.agreement, include_groups=False)
                     for q, d in chips.items()}, names=["quarter"])
per_sub = per_sub.drop(columns="neither")
per_sub[whole] = per_sub[whole].astype(int)
per_sub

## 3. Scores of chips both detected

*Gap* = |our confidence − AMW's|. About 1e-7 means the same images went in. Our quarter keeps
the last day (31 Mar, 30 Jun, 30 Sep), AMW's drops it (`referee_audit.md`), so chips under a
clear last-day scene are expected to differ.

In [ ]:
display(pd.DataFrame({q: compare.score_gap(d) for q, d in chips.items()}).T)
fig, axes = plt.subplots(1, len(QUARTERS), figsize=(4.5 * len(QUARTERS), 4), squeeze=False)
for ax, q in zip(axes[0], QUARTERS):
    d = chips[q]
    ax.scatter(d["amw_confidence"], d["confidence"], s=2, alpha=0.3)
    ax.plot([0, 1], [0, 1], "k--", lw=0.6)
    ax.axhline(0.4, color="grey", lw=0.6); ax.axvline(0.4, color="grey", lw=0.6)
    ax.set(title=q, xlabel="AMW confidence", ylabel="our confidence", xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()

In [ ]:
for q, d in chips.items():
    dis = d[d["outcome"].isin(["only ours", "only AMW"])]
    near = ((dis["confidence"] - 0.4).abs() <= 0.05).sum()
    print(f"{q}: {len(dis):,} disagreements, {near:,} with our score within 0.35-0.45")

## 4. Our months against AMW's quarter

How many of the quarter's three months detect a chip. A mine visible all quarter should be
detected in most months; clouds make single months miss. *any month* is the union of the three.

In [ ]:
rows = {}
for q, d in chips.items():
    scored = int(d["months_scored"].max())
    if scored == 0:
        print(f"{q}: no months scored, skipped")
        continue
    amw = d["amw_confidence"] > 0.4
    ours = d["confidence"] > 0.4
    anym = d["months_detected"] > 0
    rows[q] = {"months scored": scored,
               "AMW total": int(amw.sum()), "any month total": int(anym.sum()),
               "AMW and any month": int((amw & anym).sum()),
               "AMW, no month": int((amw & ~anym).sum()),
               "any month, not AMW": int((anym & ~amw).sum()),
               "overlap AMW / any month": (amw & anym).sum() / (amw | anym).sum(),
               "overlap our quarter / any month": (ours & anym).sum() / (ours | anym).sum()}
display(pd.DataFrame(rows).T)
# AMW's detections by how many of our months detected them (0 to 3)
pd.DataFrame({q: chips[q].loc[chips[q]["amw_confidence"] > 0.4, "months_detected"]
                 .value_counts().reindex(range(4), fill_value=0)
              for q in rows}).rename_axis("months detected")

## 5. Map: where our quarter and AMW's differ

In [ ]:
fig, axes = plt.subplots(1, len(QUARTERS), figsize=(6 * len(QUARTERS), 5), squeeze=False)
for ax, q in zip(axes[0], QUARTERS):
    d = chips[q].assign(gap=(chips[q]["confidence"] - chips[q]["amw_confidence"]).abs())
    same = d[d["gap"] <= 1e-6]
    differ = d[d["gap"] > 1e-6]
    ax.scatter(same["amw_key_x"], same["amw_key_y"], s=1, c="lightgrey", label=f"identical ({len(same):,})")
    sc = ax.scatter(differ["amw_key_x"], differ["amw_key_y"], s=2, c=differ["gap"], cmap="viridis",
                    vmin=0, vmax=0.05, label=f"gap > 0 ({len(differ):,})")
    for label, marker in (("only ours", "^"), ("only AMW", "v")):
        o = d[d["outcome"] == label]
        ax.scatter(o["amw_key_x"], o["amw_key_y"], s=12, marker=marker, facecolors="none",
                   edgecolors="red", lw=0.5, label=f"{label} ({len(o):,})")
    ax.set(title=q, xlabel="longitude", ylabel="latitude")
    ax.set_aspect("equal")
    ax.legend(fontsize=7, loc="best")
fig.colorbar(sc, ax=axes[0].tolist(), label="gap", shrink=0.7)

## 6. Summary

In [ ]:
for q in QUARTERS:
    c, g = summary.loc[q].to_dict(), compare.score_gap(chips[q])
    print(f"{q}: ours {c['ours total']:,.0f}, AMW {c['AMW total']:,.0f}; both {c['both']:,.0f}, "
          f"only ours {c['only ours']:,.0f}, only AMW {c['only AMW']:,.0f} (overlap {c['overlap (Jaccard)']:.3f}); "
          f"scores identical for {g['identical (gap <= 1e-6)']:.0%}, largest gap {g['largest gap']:.3f}")